# Colab Training Runbook

GPU launcher for thesis experiments. Use `single` for one EGNN/HGNN run, or `noise_sweep` for the overnight EGNN input-noise sweep.

## Before Running

1. Select a GPU runtime: `Runtime > Change runtime type > GPU`.
2. Put `train.h5`, `val.h5`, and `test.h5` under `MyDrive/masters-thesis/data/scaling/`.
3. Paste a GitHub token in Step 1 if the repo is private.
4. Keep `GIT_REF` on the branch that contains the current evaluator schema.

## Overnight Noise Sweep

Default settings run EGNN at `N_TRAIN=1000`, `EPOCHS=80`, and noise factors `0.075,0.125,0.15,0.20`.

Outputs are persisted under Drive:

```text
MyDrive/masters-thesis/runs/noise_sweep/egnn/n<N_TRAIN>_e<EPOCHS>/nf_<noise>/<run_id>/
```

Each completed run writes `best.pt`, `latest.pt`, `metrics.csv`, `evaluation/summary.csv`, and `evaluation/metrics.json`.


In [ ]:
# @title 1. Setup: Drive, experiment, and repo
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")


def _shell(*cmd: str) -> None:
    """Run a command, streaming output live to the cell, raising on failure."""
    subprocess.run(cmd, check=True)


GITHUB_TOKEN = ""  # @param {type:"string"}
REPO = "AlexNeagu123/msc-thesis-gnn-nbody"
GIT_REF = "main"  # @param {type:"string"}
DRIVE = Path("/content/drive/MyDrive/masters-thesis")

RUN_MODE = "single"  # @param ["noise_sweep", "single", "finetune"]
MODEL = "egnn_accel"  # @param ["egnn", "egnn_accel", "hgnn"]
BASE_CONFIG = ""  # @param {type:"string"}
INIT_CHECKPOINT = ""  # @param {type:"string"}
EXPERIMENT_NAME = ""  # @param {type:"string"}
N_TRAIN = "5000"  # @param ["1000", "2000", "5000", "10000"]
EPOCHS = 80  # @param {type:"integer"}
NOISE_FACTORS = "0.075,0.125,0.15,0.20"  # @param {type:"string"}
RUN_TRAINING = True  # @param {type:"boolean"}
RUN_EVALUATION = True  # @param {type:"boolean"}
SKIP_COMPLETED = False  # @param {type:"boolean"}

N_TRAIN = int(N_TRAIN)
NOISE_VALUES = [float(x.strip()) for x in NOISE_FACTORS.split(",") if x.strip()]

assert RUN_MODE in {"single", "noise_sweep", "finetune"}, RUN_MODE
assert MODEL in {"egnn", "egnn_accel", "hgnn"}, MODEL
assert EPOCHS > 0, EPOCHS
assert N_TRAIN in (1000, 2000, 5000, 10000), N_TRAIN
if RUN_MODE == "noise_sweep":
    assert MODEL == "egnn", "noise_sweep is intended for EGNN only"
    assert NOISE_VALUES, "NOISE_FACTORS must contain at least one value"
if RUN_MODE == "finetune":
    assert MODEL == "egnn", "multi-step finetune configs are currently EGNN-only"
    assert BASE_CONFIG, "BASE_CONFIG is required for finetune mode"
    assert INIT_CHECKPOINT, "INIT_CHECKPOINT is required for finetune mode"
    assert EXPERIMENT_NAME, "EXPERIMENT_NAME is required for finetune mode"
if not GITHUB_TOKEN:
    raise ValueError("Paste a GitHub token before cloning the repo.")

repo_dir = Path("/content/repo")
clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"

if (repo_dir / ".git").exists():
    _shell("git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", GIT_REF)
    _shell("git", "-C", str(repo_dir), "checkout", "FETCH_HEAD")
elif repo_dir.exists():
    raise FileExistsError(f"{repo_dir} exists but is not a git checkout")
else:
    _shell("git", "clone", "--depth", "1", "--branch", GIT_REF, clone_url, str(repo_dir))

project_dir = repo_dir / "impl" if (repo_dir / "impl" / "requirements.txt").exists() else repo_dir
if not (project_dir / "requirements.txt").exists():
    raise FileNotFoundError(f"Could not find requirements.txt under {repo_dir} or {repo_dir / 'impl'}")

get_ipython().run_line_magic("cd", str(project_dir))
print(f"Project directory: {project_dir}")
print(f"Run mode: {RUN_MODE}")
print(f"Model: {MODEL}")
if RUN_MODE == "noise_sweep":
    print(f"Noise values: {NOISE_VALUES}")
elif RUN_MODE == "finetune":
    print(f"Base config: {BASE_CONFIG}")
    print(f"Init checkpoint: {INIT_CHECKPOINT}")
    print(f"Experiment: {EXPERIMENT_NAME}")

In [ ]:
# @title 2. Install dependencies and verify GPU
_shell("pip", "install", "-q", "-r", "requirements.txt")

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Select a GPU runtime before continuing.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# @title 3. Copy scaling dataset from Drive and verify shapes
import h5py
import shutil

data_dir = Path("data/output/scaling")
data_dir.mkdir(parents=True, exist_ok=True)

for name in ["train.h5", "val.h5", "test.h5"]:
    src = DRIVE / "data" / "scaling" / name
    dst = data_dir / name
    if not src.exists():
        raise FileNotFoundError(f"Missing Drive data file: {src}")
    shutil.copy2(src, dst)

for name in ["train.h5", "val.h5", "test.h5"]:
    with h5py.File(data_dir / name, "r") as f:
        shape = f["trajectories"].shape
        print(f"{name}: {shape}")
        if name == "train.h5" and shape[0] < N_TRAIN:
            raise ValueError(f"train.h5 has {shape[0]} trajectories, but N_TRAIN={N_TRAIN}")


In [ ]:
# @title 4. Build run plan and write Colab configs
import yaml


def _noise_label(value: float) -> str:
    return f"{value:g}".replace(".", "p")


def _run_root(model_name: str, noise_factor: float | None) -> Path:
    if RUN_MODE == "noise_sweep":
        return (
            DRIVE
            / "runs"
            / "noise_sweep"
            / model_name
            / f"n{N_TRAIN}_e{EPOCHS}"
            / f"nf_{_noise_label(noise_factor or 0.0)}"
        )
    if RUN_MODE == "finetune":
        return DRIVE / "runs" / "multistep_finetune" / EXPERIMENT_NAME / f"n{N_TRAIN}_e{EPOCHS}"
    return DRIVE / "runs" / model_name / f"n{N_TRAIN}"


def _write_colab_config(
    model_name: str,
    run_root: Path,
    noise_factor: float | None,
    base_config: str | None,
) -> Path:
    cfg_path = Path(base_config) if base_config else Path(f"configs/{model_name}.yaml")
    cfg = yaml.safe_load(cfg_path.read_text())
    cfg["data"]["train_path"] = "data/output/scaling/train.h5"
    cfg["data"]["val_path"] = "data/output/scaling/val.h5"
    cfg["training"]["epochs"] = int(EPOCHS)
    cfg["training"]["device"] = "auto"
    if noise_factor is not None:
        cfg["training"]["noise_factor"] = float(noise_factor)
    cfg["checkpointing"]["enabled"] = True
    cfg["checkpointing"]["dir"] = str(run_root)
    cfg["logging"]["enabled"] = True
    cfg["logging"]["dir"] = str(run_root)

    label = EXPERIMENT_NAME if RUN_MODE == "finetune" else f"nf_{_noise_label(noise_factor or 0.0)}"
    out_path = Path("configs") / f"_colab_{model_name}_n{N_TRAIN}_e{EPOCHS}_{label}.yaml"
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return out_path


def _build_run_specs() -> list[dict]:
    """Build the list of runs to execute, dispatching on RUN_MODE."""
    if RUN_MODE == "noise_sweep":
        return [
            {
                "model": MODEL,
                "noise_factor": nf,
                "run_root": _run_root(MODEL, nf),
                "init_checkpoint": None,
                "base_config": None,
            }
            for nf in NOISE_VALUES
        ]
    if RUN_MODE == "finetune":
        cfg0 = yaml.safe_load(Path(BASE_CONFIG).read_text())
        nf = float(cfg0.get("training", {}).get("noise_factor", 0.0))
        return [
            {
                "model": MODEL,
                "noise_factor": nf,
                "run_root": _run_root(MODEL, None),
                "init_checkpoint": Path(INIT_CHECKPOINT),
                "base_config": BASE_CONFIG,
            }
        ]
    # single
    cfg0 = yaml.safe_load(Path(f"configs/{MODEL}.yaml").read_text())
    nf = float(cfg0.get("training", {}).get("noise_factor", 0.0))
    return [
        {
            "model": MODEL,
            "noise_factor": nf,
            "run_root": _run_root(MODEL, None),
            "init_checkpoint": None,
            "base_config": None,
        }
    ]


RUN_SPECS = _build_run_specs()

for spec in RUN_SPECS:
    spec["run_root"].mkdir(parents=True, exist_ok=True)
    spec["config_path"] = _write_colab_config(
        spec["model"],
        spec["run_root"],
        spec["noise_factor"] if RUN_MODE == "noise_sweep" else None,
        spec["base_config"],
    )
    print(f"noise={spec['noise_factor']}: {spec['run_root']}")
    print(f"config={spec['config_path']}")
    if spec["init_checkpoint"] is not None:
        if not spec["init_checkpoint"].exists():
            raise FileNotFoundError(f"Missing init checkpoint: {spec['init_checkpoint']}")
        print(f"init_checkpoint={spec['init_checkpoint']}")

In [ ]:
# @title 5. Train and evaluate planned runs
RUN_RESULTS = []


def _train_cmd(config_path: Path, init_checkpoint: Path | None) -> list[str]:
    cmd = [
        "python", "-u", "-m", "training.train",
        "--config", str(config_path),
        "--n-train", str(N_TRAIN),
    ]
    if init_checkpoint is not None:
        cmd += ["--init-checkpoint", str(init_checkpoint)]
    return cmd


def _eval_cmd(config_path: Path, checkpoint: Path, eval_dir: Path) -> list[str]:
    return [
        "python", "-u", "-m", "evaluation.evaluate",
        "--config", str(config_path),
        "--checkpoint", str(checkpoint),
        "--test-path", "data/output/scaling/test.h5",
        "--output-dir", str(eval_dir),
        "--device", "cuda",
    ]


for index, spec in enumerate(RUN_SPECS, start=1):
    model_name = spec["model"]
    noise_factor = spec["noise_factor"]
    run_root = spec["run_root"]
    config_path = spec["config_path"]
    init_checkpoint = spec.get("init_checkpoint")
    label = EXPERIMENT_NAME if RUN_MODE == "finetune" else _noise_label(noise_factor)

    existing = sorted(
        p for p in run_root.iterdir()
        if p.is_dir() and (p / "evaluation" / "metrics.json").exists()
    )
    if SKIP_COMPLETED and existing:
        run_dir = existing[-1]
        print(f"[{index}/{len(RUN_SPECS)}] skipping completed {label}: {run_dir}")
    elif RUN_TRAINING:
        print(
            f"[{index}/{len(RUN_SPECS)}] training {model_name} "
            f"N={N_TRAIN}, epochs={EPOCHS}, noise={noise_factor}; output={run_root}",
            flush=True,
        )
        _shell(*_train_cmd(config_path, init_checkpoint))
        run_ids = sorted(
            p.name for p in run_root.iterdir() if p.is_dir() and (p / "latest.pt").exists()
        )
        if not run_ids:
            raise RuntimeError(f"No checkpoint runs found under {run_root}.")
        run_dir = run_root / run_ids[-1]
    else:
        run_ids = sorted(
            p.name for p in run_root.iterdir() if p.is_dir() and (p / "latest.pt").exists()
        )
        if not run_ids:
            raise RuntimeError(f"Training skipped and no checkpoint runs found under {run_root}.")
        run_dir = run_root / run_ids[-1]

    result = {
        "model": model_name,
        "noise_factor": noise_factor,
        "noise_label": label,
        "config_path": config_path,
        "run_root": run_root,
        "run_dir": run_dir,
        "best_checkpoint": run_dir / "best.pt",
        "latest_checkpoint": run_dir / "latest.pt",
        "eval_dir": run_dir / "evaluation",
    }
    RUN_RESULTS.append(result)
    print(f"ready: {label}, run_dir={run_dir}")

    if RUN_EVALUATION:
        print(
            f"[{index}/{len(RUN_SPECS)}] evaluating {label} "
            f"checkpoint={result['best_checkpoint']}",
            flush=True,
        )
        _shell(*_eval_cmd(config_path, result["best_checkpoint"], result["eval_dir"]))
    else:
        print(f"Evaluation skipped for {label}.")

In [ ]:
# @title 6. Print evaluation summaries
for index, result in enumerate(RUN_RESULTS, start=1):
    summary_path = result["eval_dir"] / "summary.csv"
    print(f"Evaluation directory: {result['eval_dir']}")
    if summary_path.exists():
        print(summary_path.read_text())


In [ ]:
# @title 7. Verify persistent Drive artifacts
for result in RUN_RESULTS:
    print(f"\nnoise={result['noise_factor']}  run={result['run_dir']}")
    items = [
        result["best_checkpoint"],
        result["latest_checkpoint"],
        result["run_dir"] / "metrics.csv",
        result["run_dir"] / "diagnostics.log",
        result["eval_dir"] / "summary.csv",
        result["eval_dir"] / "metrics.json",
    ]

    for path in items:
        status = "ok" if path.exists() else "missing"
        print(f"{status}: {path}")

print("\nAll planned run directories:")
for result in RUN_RESULTS:
    print(result["run_dir"])
